# Families

A passenger with SibSp + Parch = 0 travelled with no family and has no family ID. A passenger with SibSp + Parch = x belongs to a family of x + 1 people, so x other passengers should share their family.

train.csv and test.csv are grouped together, because Kaggle split the passenger list at random and many families have members in both files. Only passengers with SibSp + Parch above 0 are grouped.

**Check.** A group is correct when its size equals SibSp + Parch + 1 for every member of it. The check tests group sizes, not who is in the group, so a correct size is strong evidence but not proof.

## Method

**Run 1: surname only.** Every passenger is grouped with every other passenger who has the same surname (the text before the comma in `Name`).

**Run 2: surname, ticket and cabin.** Passengers are joined in this order, strongest evidence first:

1. Same surname and same ticket
2. Same ticket
3. Same cabin. A passenger with several cabins, such as `C23 C25 C27`, is joined to anyone sharing any one of them
4. Same surname

A join is made only when the combined group is no bigger than SibSp + Parch + 1 for every person in it. This stops a common surname such as Andersson from swallowing unrelated families, and stops a shared ticket from pulling in more people than the family says it has.

After that, any groups still failing the check that share a ticket or a cabin are joined regardless of size and marked **Linked, sizes disagree**.

In [1]:
import pandas as pd
from IPython.display import display, Markdown

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
people = pd.concat([train, test], ignore_index=True).sort_values("PassengerId", ignore_index=True)
people["Surname"] = people["Name"].str.split(",").str[0].str.strip()
people["Expected"] = people["SibSp"] + people["Parch"] + 1
fam = people[people["Expected"] > 1].copy()

cabins = (
    fam["Cabin"].dropna().str.split().explode().rename("CabinNo").reset_index()
    .rename(columns={"index": "row"})
)

def groups(keys):
    """Lists of row labels that share the given key(s)."""
    if keys == "Cabin":
        return [list(g["row"]) for _, g in cabins.groupby("CabinNo") if len(g) > 1]
    return [list(ix) for ix in fam.groupby(keys).groups.values() if len(ix) > 1]

class Families:
    def __init__(self, rows):
        self.parent = {r: r for r in rows}
        self.size = {r: 1 for r in rows}
        self.cap = {r: fam.at[r, "Expected"] for r in rows}
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def join(self, a, b, capped=True):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if capped and self.size[ra] + self.size[rb] > min(self.cap[ra], self.cap[rb]):
            return
        self.parent[ra] = rb
        self.size[rb] += self.size[ra]
        self.cap[rb] = min(self.cap[ra], self.cap[rb])
    def labels(self):
        return pd.Series({r: self.find(r) for r in self.parent})

def check(label):
    g = fam.assign(G=label).groupby("G")["Expected"].agg(["size", "min", "max"])
    return ((g["min"] == g["max"]) & (g["size"] == g["min"])).rename("Correct")

# Run 1: surname only
run1 = Families(fam.index)
for ix in groups("Surname"):
    for r in ix[1:]:
        run1.join(r, ix[0], capped=False)
fam["Run1"] = run1.labels()

# Run 2: surname + ticket, ticket, cabin, surname, each capped at the family size
run2 = Families(fam.index)
for keys in [["Surname", "Ticket"], "Ticket", "Cabin", "Surname"]:
    for ix in groups(keys):
        for r in ix[1:]:
            run2.join(r, ix[0])
lab = run2.labels()
ok = check(lab)
correct_rows = set(lab[lab.map(ok)].index)

# Second pass: join still-failing groups that share a ticket or cabin, ignoring size
for keys in ["Ticket", "Cabin"]:
    for ix in groups(keys):
        ix = [r for r in ix if r not in correct_rows]
        for r in ix[1:]:
            run2.join(r, ix[0], capped=False)
fam["Run2"] = run2.labels()

sizes = fam.groupby("Run2")["PassengerId"].transform("size")
fam["FamilyCheck"] = "Correct"
fam.loc[~fam.index.isin(correct_rows) & (sizes > 1), "FamilyCheck"] = "Linked, sizes disagree"
fam.loc[~fam.index.isin(correct_rows) & (sizes == 1), "FamilyCheck"] = "No relative found"

# FamilyId from 1, in order of each family's first PassengerId
first = fam.groupby("Run2")["PassengerId"].min().sort_values()
fam["FamilyId"] = fam["Run2"].map(pd.Series(range(1, len(first) + 1), index=first.index))
people = people.join(fam[["FamilyId", "FamilyCheck"]])
people["FamilyId"] = people["FamilyId"].astype("Int64")

def with_family(src):
    cols = list(src.columns)
    at = cols.index("Parch") + 1
    cols[at:at] = ["FamilyId", "FamilyCheck"]
    return src.merge(people[["PassengerId", "FamilyId", "FamilyCheck"]], on="PassengerId")[cols]

train_out, test_out = with_family(train), with_family(test)
train_out.to_csv("data/train_families.csv", index=False)
test_out.to_csv("data/test_families.csv", index=False)

GREEN = "#d9f2d0"
with pd.ExcelWriter("data/families.xlsx", engine="openpyxl") as xl:
    for sheet, out in [("train", train_out), ("test", test_out)]:
        out.style.set_properties(subset=["FamilyId", "FamilyCheck"], **{"background-color": GREEN}) \
            .to_excel(xl, sheet_name=sheet, index=False)
        xl.sheets[sheet].freeze_panes = "A2"

n = len(fam)
r1 = int(fam["Run1"].map(check(fam["Run1"])).sum())
r2 = len(correct_rows)
display(Markdown(
    f"{len(people):,} passengers. {len(people) - n:,} travelled alone and have no FamilyId. {n:,} have family aboard.\n\n"
    f"**Run 1 (surname): {r1} of {n} correct ({100 * r1 / n:.0f}%). "
    f"Run 2 (surname, ticket and cabin): {r2} of {n} correct ({100 * r2 / n:.0f}%).**"
))

1,309 passengers. 790 travelled alone and have no FamilyId. 519 have family aboard.

**Run 1 (surname): 401 of 519 correct (77%). Run 2 (surname, ticket and cabin): 445 of 519 correct (86%).**

## Results

In [2]:
def tally(label, verdict):
    t = pd.DataFrame({"G": label, "V": verdict}).groupby("V").agg(
        Groups=("G", "nunique"), Passengers=("G", "size"))
    return t

r1_ok = fam["Run1"].map(check(fam["Run1"]))
r1_size = fam.groupby("Run1")["PassengerId"].transform("size")
r1_v = pd.Series("Correct", index=fam.index)
r1_v[~r1_ok & (r1_size > 1)] = "Linked, sizes disagree"
r1_v[~r1_ok & (r1_size == 1)] = "No relative found"

order = ["Correct", "Linked, sizes disagree", "No relative found"]
t = pd.concat({"Run 1: surname": tally(fam["Run1"], r1_v), "Run 2: surname, ticket, cabin": tally(fam["Run2"], fam["FamilyCheck"])}, axis=1)
t = t.reindex(order).fillna(0).astype(int)
t.loc["Total"] = t.sum()
t.index.name = None
t

Run 1: surname             \
                               Groups Passengers   
Correct                           150        401   
Linked, sizes disagree             30         89   
No relative found                  29         29   
Total                             209        519   

                       Run 2: surname, ticket, cabin             
                                              Groups Passengers  
Correct                                          167        445  
Linked, sizes disagree                            25         54  
No relative found                                 20         20  
Total                                            212        519

## Relatives found by ticket or cabin

Families whose members have more than one surname. Surname alone could not have put these people together. Every one of them passes the check.

In [3]:
mixed = (
    fam[fam["FamilyCheck"] == "Correct"].groupby("FamilyId")
    .filter(lambda g: g["Surname"].nunique() > 1)
    .sort_values(["FamilyId", "PassengerId"])
)
display(Markdown(f"{mixed['FamilyId'].nunique()} families, {len(mixed)} passengers."))
mixed[["FamilyId", "PassengerId", "Name", "SibSp", "Parch", "Ticket", "Cabin"]] \
    .style.format(na_rep="").hide(axis="index")

9 families, 20 passengers.

FamilyId,PassengerId,Name,SibSp,Parch,Ticket,Cabin
57,146,"Nicholls, Mr. Joseph Charles",1,1,C.A. 33112,
57,550,"Davies, Master. John Morgan Jr",1,1,C.A. 33112,
57,1222,"Davies, Mrs. John Morgan (Elizabeth Agnes Mary White)",0,2,C.A. 33112,
65,167,"Chibnall, Mrs. (Edith Martha Bowerman)",0,1,113505,E33
65,357,"Bowerman, Miss. Elsie Edith",0,1,113505,E33
89,260,"Parrish, Mrs. (Lutie Davis)",0,1,230433,
89,881,"Shelley, Mrs. William (Imanita Parrish Hall)",0,1,230433,
95,276,"Andrews, Miss. Kornelia Theodosia",1,0,13502,D7
95,766,"Hogeboom, Mrs. John C (Anna Andrews)",1,0,13502,D11
138,497,"Eustis, Miss. Elizabeth Mussey",1,0,36947,D20


## Still not resolved

**Linked, sizes disagree.** These passengers share a ticket, a cabin or a surname, but their SibSp + Parch counts cannot all be true for one closed group. Most are households where the counts are not the same for everyone: a husband counts his wife and his own siblings, while his wife counts only him (Vander Planke), or a mother counts her children and her own parents and siblings travelling separately (Richards, née Hocking). These are likely real families that the "x + 1 people" rule cannot describe.

**No relative found.** No other passenger with family aboard shares their surname, ticket or cabin within the size limit. Several are married women whose relatives travel under her maiden name, which appears in brackets in `Name`: Wilkes (Ellen Needs) is sister to Hocking (Eliza Needs), and Renouf (Lillian Jefferys) is sister to the Jefferys brothers. Matching the bracketed maiden name to a surname would recover some of these.

In [4]:
rest = fam[fam["FamilyCheck"] != "Correct"].sort_values(["FamilyCheck", "FamilyId", "PassengerId"])
rest[["FamilyCheck", "FamilyId", "PassengerId", "Name", "SibSp", "Parch", "Expected", "Ticket", "Cabin"]] \
    .rename(columns={"Expected": "Family size"}).style.format(na_rep="").hide(axis="index")

FamilyCheck,FamilyId,PassengerId,Name,SibSp,Parch,Family size,Ticket,Cabin
"Linked, sizes disagree",10,19,"Vander Planke, Mrs. Julius (Emelia Maria Vandemoortele)",1,0,2,345763,
"Linked, sizes disagree",10,1037,"Vander Planke, Mr. Julius",3,0,4,345763,
"Linked, sizes disagree",16,39,"Vander Planke, Miss. Augusta Maria",2,0,3,345764,
"Linked, sizes disagree",16,334,"Vander Planke, Mr. Leo Edmondus",2,0,3,345764,
"Linked, sizes disagree",34,70,"Kink, Mr. Vincenz",2,0,3,315151,
"Linked, sizes disagree",34,1268,"Kink, Miss. Maria",2,0,3,315152,
"Linked, sizes disagree",37,86,"Backstrom, Mrs. Karl Alfred (Maria Mathilda Gustafsson)",3,0,4,3101278,
"Linked, sizes disagree",37,207,"Backstrom, Mr. Karl Alfred",1,0,2,3101278,
"Linked, sizes disagree",45,105,"Gustafsson, Mr. Anders Vilhelm",2,0,3,3101276,
"Linked, sizes disagree",45,393,"Gustafsson, Mr. Johan Birger",2,0,3,3101277,


In [5]:
display(Markdown("## train_families.csv, first 25 rows\n\nFamilyId and FamilyCheck are the light green columns."))
train_out.head(25).style.set_properties(subset=["FamilyId", "FamilyCheck"], **{"background-color": GREEN}) \
    .format(na_rep="", precision=2).hide(axis="index")

## train_families.csv, first 25 rows

FamilyId and FamilyCheck are the light green columns.

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,FamilyId,FamilyCheck,Ticket,Fare,Cabin,Embarked
1,0,3,"Braund, Mr. Owen Harris",male,22.00,1,0,1,Correct,A/5 21171,7.25,,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38.00,1,0,2,Correct,PC 17599,71.28,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.00,0,0,,,STON/O2. 3101282,7.92,,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.00,1,0,3,Correct,113803,53.10,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.00,0,0,,,373450,8.05,,S
6,0,3,"Moran, Mr. James",male,,0,0,,,330877,8.46,,Q
7,0,1,"McCarthy, Mr. Timothy J",male,54.00,0,0,,,17463,51.86,E46,S
8,0,3,"Palsson, Master. Gosta Leonard",male,2.00,3,1,4,Correct,349909,21.07,,S
9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.00,0,2,5,Correct,347742,11.13,,S
10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.00,1,0,6,Correct,237736,30.07,,C


## Files written

- `data/train_families.csv` and `data/test_families.csv`: the original columns with `FamilyId` and `FamilyCheck` added straight after `Parch`, both blank for passengers with no family. `FamilyId` comes from run 2
- `data/families.xlsx`: the same data on a train sheet and a test sheet, with the two new columns shaded light green. A CSV is plain text and cannot hold a colour, so the shading lives in the xlsx

The original train.csv and test.csv are unchanged.